# Merged notebook

This notebook merges together all of our code used during the project -- except testing notebooks (such as unit tests and debuging)

## Early attempt to train

In [ ]:
import random
from typing import List, Tuple
import json
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
import io
import time
import os

In [ ]:
def format_query(query: str, prefix: str = '') -> str:
    return f'{prefix} {query.strip()}'.strip()

def format_passage(text: str, title: str = '', prefix: str = '') -> str:
    return f'{prefix} {title.strip()} {text.strip()}'.strip()

def load_jsonl(file_path):
    """Load JSONL file into a list of dictionaries."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

class TrainDataset(Dataset):
    def __init__(self, dataset_domain=["ArxivQA", "DUDE_long", "SciQAG", "SlideVQA", "TAT-DQA", "Wiki-ss"],
                 base_dir="."):
        self.train_data = []
        self.page_image_df = None
        self.dataset_domain = dataset_domain
        self.base_dir = base_dir

        for i, domain in enumerate(self.dataset_domain):
            start_time = time.time()  # Record the start time

            # Path to local parquet file
            parquet_file = os.path.join(self.base_dir, "parquet", f"{domain}_filter.parquet")

            # Read parquet file
            df = pd.read_parquet(parquet_file)
            if (i == 0):
                self.page_image_df = df
            else:
                self.page_image_df = pd.concat([self.page_image_df, df], ignore_index=False)

            # Path to local JSONL file
            json_file = os.path.join(self.base_dir, "annotations_top1_negative", f"annotations_top1_negative%2F{domain}_train.jsonl")

            # Replace load_dataset with local file loading
            sub_train_data = load_jsonl(json_file)
            self.train_data.extend(sub_train_data)

            end_time = time.time()  # Record the end time
            time_taken = end_time - start_time
            print(f"-----reading {domain}_filter.parquet takes {time_taken} seconds-----")

    def __len__(self):
        return len(self.train_data)

    def _get_image(self, doc_name, page_id):
        item_row = self.page_image_df[
            (self.page_image_df['file_name'] == doc_name) & (self.page_image_df['page'] == page_id)]
        if len(item_row) == 1:
            img_bytes, page_size, page_layouts = item_row["image"].iloc[0], item_row["page_size"].iloc[0], \
            item_row["layouts"].iloc[0]
            image = Image.open(io.BytesIO(img_bytes))
            return {"image": image, "page_size": page_size, "page_layouts": page_layouts, "file_name": doc_name,
                    "page_id": page_id}
        else:
            raise ValueError(f"Document {doc_name} does not have page {page_id}! Please check your data")

    def __getitem__(self, item) -> Tuple[str, List[str]]:
        group = self.train_data[item]
        query = group['query']
        group_positives, group_negatives = group['positive_passages'], group['negative_passages']
        formated_query = format_query(query, "")
        pos_psg, neg_psg = group_positives[0], group_negatives[0]
        formated_passages = [self._get_image(pos_psg['doc_name'], pos_psg['page_id'])]
        formated_passages.append(self._get_image(neg_psg['doc_name'], neg_psg['page_id']))
        return formated_query, formated_passages

In [ ]:

base_directory = ".."
train_dataset = TrainDataset(
    dataset_domain=["ArxivQA", "DUDE_long", "SciQAG", "SlideVQA", "TAT-DQA", "Wiki-ss"],
    base_dir=base_directory
)
print(f"There are {train_dataset.__len__()} instances in the training dataset.")

formated_query, formated_passages = train_dataset.__getitem__(50)
print(formated_query)
print(formated_passages)

## colpali

colpali + trying memory saving techniques

In [ ]:
import random
from typing import List, Tuple
import json
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
import io
import time
import os
from pyarrow.parquet import ParquetFile
import pyarrow as pa
import dask.dataframe as dd


def format_query(query: str, prefix: str = '') -> str:
    return f'{prefix} {query.strip()}'.strip()

def format_passage(text: str, title: str = '', prefix: str = '') -> str:
    return f'{prefix} {title.strip()} {text.strip()}'.strip()

def load_jsonl(file_path):
    """Load JSONL file into a list of dictionaries."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

class TrainDataset(Dataset):
    def __init__(self, dataset_domain=["ArxivQA", "DUDE_long", "SciQAG", "SlideVQA", "TAT-DQA", "Wiki-ss"],
                 base_dir="."):
        self.train_data = []
        self.page_image_df = None
        self.dataset_domain = dataset_domain
        self.base_dir = base_dir

        for i, domain in enumerate(self.dataset_domain):
            start_time = time.time()  # Record the start time

            # Path to local parquet file
            parquet_file = os.path.join(self.base_dir, "parquet", f"{domain}_filter.parquet")


            # # Read parquet file -- basic -- memory error
            # df = pd.read_parquet(parquet_file, engine='pyarrow')


            ddf = dd.read_parquet(parquet_file, columns=["file_name"])
            doc_df = ddf.compute()  # Only contains "file_name"
            unique_docs = doc_df["file_name"].unique()  # Get all unique

            chunk_size = 30  # Number of documents per chunk
            for j in range(0, len(unique_docs), chunk_size):
                doc_chunk = unique_docs[j:j + chunk_size]  # Select a subset of documents
                # Load only the relevant rows from the full dataset
                df_chunk = pd.read_parquet(parquet_file, filters=[("file_name", "in", list(doc_chunk))])
                # Process df_chunk here
                print(f"Processing chunk {j // chunk_size + 1} with {len(df_chunk)} rows")
                if (i == 0):
                    self.page_image_df = df_chunk
                else:
                    self.page_image_df = pd.concat([self.page_image_df, df_chunk], ignore_index=False)



            # Path to local JSONL file
            json_file = os.path.join(self.base_dir, "annotations_top1_negative", f"annotations_top1_negative%2F{domain}_train.jsonl")

            # Replace load_dataset with local file loading
            sub_train_data = load_jsonl(json_file)
            self.train_data.extend(sub_train_data)

            end_time = time.time()  # Record the end time
            time_taken = end_time - start_time
            print(f"-----reading {domain}_filter.parquet takes {time_taken} seconds-----")

    def __len__(self):
        return len(self.train_data)

    def _get_image(self, doc_name, page_id):
        item_row = self.page_image_df[
            (self.page_image_df['file_name'] == doc_name) & (self.page_image_df['page'] == page_id)]
        if len(item_row) == 1:
            img_bytes, page_size, page_layouts = item_row["image"].iloc[0], item_row["page_size"].iloc[0], \
            item_row["layouts"].iloc[0]
            image = Image.open(io.BytesIO(img_bytes))
            return {"image": image, "page_size": page_size, "page_layouts": page_layouts, "file_name": doc_name,
                    "page_id": page_id}
        else:
            raise ValueError(f"Document {doc_name} does not have page {page_id}! Please check your data")

    def __getitem__(self, item) -> Tuple[str, List[str]]:
        group = self.train_data[item]
        query = group['query']
        group_positives, group_negatives = group['positive_passages'], group['negative_passages']
        formated_query = format_query(query, "")
        pos_psg, neg_psg = group_positives[0], group_negatives[0]
        formated_passages = [self._get_image(pos_psg['doc_name'], pos_psg['page_id'])]
        formated_passages.append(self._get_image(neg_psg['doc_name'], neg_psg['page_id']))
        return formated_query, formated_passages

In [ ]:
base_directory = ".."  # Up one level
train_dataset = TrainDataset(
    dataset_domain=["ArxivQA", "DUDE_long", "SciQAG", "SlideVQA", "TAT-DQA", "Wiki-ss"],
    base_dir=base_directory
)
print(f"There are {train_dataset.__len__()} instances in the training dataset.")
# Load the query and passage of index 50 from the training dataset.
formated_query, formated_passages = train_dataset.__getitem__(50)
print(formated_query)
print(formated_passages)

In [ ]:
for n in range(train_dataset.__len__()):
    if n % 10000 == 0:
        formated_query,formated_passages = train_dataset.__getitem__(n)
        print(n, formated_query)
# formated_query, formated_passages = train_dataset.__getitem__(4)
# print(formated_query)
# print(formated_passages)

In [ ]:
## Reference from challenge

from typing import cast
import torch
from PIL import Image
from colpali_engine.models import ColPali, ColPaliProcessor

model_name = "vidore/colpali-v1.3"

model = ColPali.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",  # or "mps" if on Apple Silicon
).eval()

processor = ColPaliProcessor.from_pretrained(model_name)

# Your inputs
images = [
    Image.new("RGB", (32, 32), color="white"),
    Image.new("RGB", (16, 16), color="black"),
]
queries = [
    "Is attention really all you need?",
    "Are Benjamin, Antoine, Merve, and Jo best friends?",
]

# Process the inputs
batch_images = processor.process_images(images).to(model.device)
batch_queries = processor.process_queries(queries).to(model.device)

# Forward pass
with torch.no_grad():
    image_embeddings = model(**batch_images)
    query_embeddings = model(**batch_queries)

scores = processor.score_multi_vector(query_embeddings, image_embeddings)


## colqwen2

We used colqwen2 because of the cookbook we found that had some good starter code.

In [ ]:
from io import BytesIO
from typing import Any, List, cast

import requests
import torch
from colpali_engine.models import ColQwen2, ColQwen2Processor
from colpali_engine.utils.torch_utils import get_torch_device
from IPython.display import display
from peft import LoraConfig
from PIL import Image
from transformers.models.qwen2_vl import Qwen2VLForConditionalGeneration, Qwen2VLProcessor

import json
import pandas as pd

In [ ]:
def split_before_last_underscore(input_string):
    last_underscore_index = input_string.rfind('_')
    return input_string[:last_underscore_index]  # Return the substring before the last underscore

def get_challege_df():
    '''
    Returns dataframes of the json files in the Challenge Dataset
    '''

    gt_remove_filepath = "..\\MMDocIR-Challege\\MMDocIR_gt_remove.jsonl"
    passages_filepath = "..\\MMDocIR-Challege\\MMDocIR_doc_passages.json"

    with open(passages_filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    flattened_data = []
    for doc_id, passages in data.items():
        for passage in passages:
            passage["doc_id"] = doc_id
            flattened_data.append(passage)

    df_passages = pd.DataFrame(flattened_data)
    df_gt = pd.read_json(gt_remove_filepath, lines=True)

    return df_passages, df_gt

def load_image_from_path(path: str) -> Image.Image:
    """
    Load a PIL image from a valid URL.
    """
    return Image.open(path)


def scale_image(image: Image.Image, new_height: int = 1024) -> Image.Image:
    """
    Scale an image to a new height while maintaining the aspect ratio.
    """
    width, height = image.size
    aspect_ratio = width / height
    new_width = int(new_height * aspect_ratio)

    scaled_image = image.resize((new_width, new_height))

    return scaled_image


def get_document(df_passages,df_gt):
    passage_list = []


    return passage_list

In [ ]:
class ColQwen2ForRAG(ColQwen2):
    """
    ColQwen2 model implementation that can be used both for retrieval and generation.
    Allows switching between retrieval and generation modes.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._is_retrieval_enabled = True

    def forward(self, *args, **kwargs) -> Any:
        """
        Forward pass that calls either Qwen2VLForConditionalGeneration.forward for generation
        or ColQwen2.forward for retrieval based on the current mode.
        """
        if self.is_retrieval_enabled:
            return ColQwen2.forward(self, *args, **kwargs)
        else:
            return Qwen2VLForConditionalGeneration.forward(self, *args, **kwargs)

    def generate(self, *args, **kwargs):
        """
        Generate text using Qwen2VLForConditionalGeneration.generate.
        """
        if not self.is_generation_enabled:
            raise ValueError(
                "Set the model to generation mode by calling `enable_generation()` before calling `generate()`."
            )
        return super().generate(*args, **kwargs)

    @property
    def is_retrieval_enabled(self) -> bool:
        return self._is_retrieval_enabled

    @property
    def is_generation_enabled(self) -> bool:
        return not self.is_retrieval_enabled

    def enable_retrieval(self) -> None:
        """
        Switch to retrieval mode.
        """
        self.enable_adapters()
        self._is_retrieval_enabled = True

    def enable_generation(self) -> None:
        """
        Switch to generation mode.
        """
        self.disable_adapters()
        self._is_retrieval_enabled = False

In [ ]:
model_name = "vidore/colqwen2-v1.0"
device = get_torch_device("auto")

print(f"Using device: {device}")

# Get the LoRA config from the pretrained retrieval model
lora_config = LoraConfig.from_pretrained(model_name)

# Load the processors
processor_retrieval = cast(ColQwen2Processor, ColQwen2Processor.from_pretrained(model_name))
processor_generation = cast(Qwen2VLProcessor, Qwen2VLProcessor.from_pretrained(lora_config.base_model_name_or_path))

# Load the model with the loaded pre-trained adapter for retrieval
model = cast(
    ColQwen2ForRAG,
    ColQwen2ForRAG.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map=device,
    ),
)

In [ ]:
df_passages, df_gt = get_challege_df()

df_passages['transformed_image_path'] = df_passages['image_path'].apply(
    lambda path: '..\\MMDocIR-Challege\\page_images\\page_images\\' + path.split('/')[-1]
)

df_passages['doc_name'] = df_passages['transformed_image_path'].apply(lambda doc: doc.split('\\')[-1]).apply(split_before_last_underscore)


for question_id in df_gt['question_id']:
    print("BEGIN processing question_id:", question_id)
    ## This is the pseudo code implementation
    image_path_list = df_passages['transformed_image_path'].where(df_passages['doc_name'] == df_gt['doc_name'].where(df_gt['question_id'] == question_id).dropna().item()).dropna().to_list()  ## this might be very slow
    images = [load_image_from_path(item) for item in image_path_list]

    images = [scale_image(image, new_height=512) for image in images]
    query = df_gt['question'].where(df_gt['question_id'] == question_id).dropna().item()

    print("Finished image loading... number of images:", len(images))
    # RAG
        # Process the inputs
    batch_images = processor_retrieval.process_images(images).to(model.device)
    batch_queries = processor_retrieval.process_queries([query]).to(model.device)

    # Forward pass
    model.enable_retrieval()

    with torch.no_grad():
        image_embeddings = model.forward(**batch_images)
        query_embeddings = model.forward(**batch_queries)

    # Calculate the similarity scores
    scores = processor_retrieval.score_multi_vector(query_embeddings, image_embeddings)

    # Get the top-5 page image
    _,retrieved_image_indexes = torch.topk(scores, 5)

    retrieved_images = images[retrieved_image_indexes]

    print(f"Top image retrieved for the following query: `{query}`")
    display(scale_image(retrieved_images[0], new_height=512))
    print("\n")



    if question_id-10000 > 3:
        break



## TF-IDF

Term Frequency - Inverse Document Frequency (simple algorithm to *at least* get *some* output)

In [ ]:
import re
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import heapq


def term_frequency(t: str, p: str) -> float:
    """
    Calculates the term frequency of a search term in a passage.

    Args:
        t: The search term string (e.g., 'source').
        p: The passage string (the text to search within).

    Returns:
        The term frequency score, calculated as the number of times
        the search term appears in the passage divided by the total
        number of terms in the passage.
    """
    # Normalize the passage by converting to lowercase and removing punctuation
    p_normalized = re.sub(r'[^\w\s]', '', p).lower()
    t_normalized = t.lower()

    # Split the normalized passage into terms
    terms = p_normalized.split()

    # Count the occurrences of the normalized search term
    t_count = terms.count(t_normalized)

    # Calculate the total number of terms in the passage
    total_terms = len(terms)

    if total_terms == 0:
        return 0.0  # Avoid division by zero if the passage is empty

    # Calculate the term frequency
    tf_score = t_count / total_terms
    return tf_score


def inverse_document_frequency(t: str, D: list) -> float:
    """
    Calculates the inverse document frequency of a search term in a given passage of a document.

    Args:
        t: The search term string (e.g., 'source').
        p: The passage string (the text to search within).
        D: The document list of passage strings

    Returns:
        The inverse document frequency score, calculated as the log of the number of passages in the document divided
        by the number of passages in the document containing the search term.
    """
    t_normalized = t.lower()
    passages_with_t = 1
    for p in D:
        # Normalize the passage by converting to lowercase and removing punctuation
        p_normalized = re.sub(r'[^\w\s]', '', p).lower()

        # Split the normalized passage into terms
        terms = p_normalized.split()

        # Count the occurrences of the normalized search term
        t_count = terms.count(t_normalized)

        if t_count > 0:
            passages_with_t += 1

    # Calculate the term frequency
    tf_score = np.log(len(D)/passages_with_t)
    return tf_score


In [ ]:
def get_challege_df():
    '''
    Returns dataframes of the json files in the Challenge Dataset
    '''

    gt_remove_filepath = "..\\MMDocIR-Challege\\MMDocIR_gt_remove.jsonl"
    passages_filepath = "..\\MMDocIR-Challege\\MMDocIR_doc_passages.json"

    with open(passages_filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    flattened_data = []
    for doc_id, passages in data.items():
        for passage in passages:
            passage["doc_id"] = doc_id
            flattened_data.append(passage)

    df_passages = pd.DataFrame(flattened_data)
    df_gt = pd.read_json(gt_remove_filepath, lines=True)

    return df_passages, df_gt

In [ ]:
df_passages, df_gt = get_challege_df()

print(df_passages.columns)
print(df_gt.columns)

In [ ]:
scores = {}
for question_id, question, doc_name in zip(df_gt['question_id'], df_gt['question'], df_gt['doc_name']):
    document = df_passages[df_passages['doc_id'] == doc_name]['vlm_text'].tolist()
    terms = question.split()
    passage_scores = []

    for i, page in enumerate(document):
        total_score = sum(term_frequency(t, page) * inverse_document_frequency(t, document) for t in terms)
        passage_scores.append((i, total_score))

    scores[question_id] = passage_scores


df_scores = pd.DataFrame([
    {"question_id": qid, "page_index": idx, "score": score}
    for qid, passages in scores.items() for idx, score in passages
])


In [ ]:
df_scores.to_json('scores.json')

df_scores.describe()

## ColBERT
This involves \
1) Preprocessing into tsv, which ColBERT can use. \
2) Precompute the ColBERT representations of the passages before doing any querying. This is the simplest way to use the model. \
3) Try to query ColBERT.

### Getting ColBERT
We also have to get ColBERT first, since it's not hosted as a package on pypi.

In [ ]:
import sys
!git -C ColBERT/ pull || git clone https://github.com/stanford-futuredata/ColBERT.git
sys.path.insert(0, 'ColBERT')

We also need to grab the pretrained model to use as our baseline.

In [ ]:
os.mkdir('downloads')
!wget https://downloads.cs.stanford.edu/nlp/data/colbert/colbertv2/colbertv2.0.tar.gz -P downloads/
!tar -xvzf downloads/colbertv2.0.tar.gz -C downloads

### Preprocess passages and queries as tsv

In [ ]:
import json
import pandas as pd
import re
import os
gt_remove = "MMDocIR-Challenge/MMDocIR_gt_remove.jsonl"
doc_passages = "MMDocIR-Challenge/MMDocIR_doc_passages.json"

with open(doc_passages) as file:
    doc_passages_dict = json.load(file)
with open(gt_remove) as file:
    gt_remove_df = pd.read_json(gt_remove, lines=True)

# save all tsv files in this folder
data_root = "./documents"; os.mkdir(data_root);

# process the passages as tsv
for doc_name, passage_list in doc_passages_dict.items():
    doc_path = os.path.join(data_root, doc_name)
    os.mkdir(doc_path)
    collection_path = os.path.join(doc_path, 'collection.tsv')
    with open(collection_path, mode='w') as file:
        for passage in passage_list:
            file.write(passage['passage_id'])
            file.write('\t')
            file.write(re.sub('\\s+', ' ', passage['ocr_text']))
            file.write('\n')

# process the queries as tsv
for question_id, question, doc_name in zip(gt_remove_df['question_id'], gt_remove_df['question'], gt_remove_df['doc_name']):
    query_path = os.path.join(data_root, doc_name, 'questions.tsv')
    with open(query_path, mode='a') as file:
        file.write(str(question_id))
        file.write('\t')
        file.write(re.sub('\n', '', question))
        file.write('\n')

### Indexing

In [ ]:
from colbert.infra import Run, RunConfig, ColBERTConfig
from colbert import Indexer


# Indexing - Precompute the ColBERT representations of passages
# Running this twice will cause it to complain - overwriting existing precomputations, delete the "experiments" folder to rerun
# This gets stuck on the 5th document - running on my available compute - waited 2 hours with no visible progress
nbits = 2
doc_maxlen = 300
checkpoint_path = 'downloads/colbertv2.0'
start_time = time.time()
for doc_name, passage_list in doc_passages_dict.items():
    collection_path = os.path.join(data_root, doc_name, 'collection.tsv')

    config = ColBERTConfig(
        nbits=nbits,
        root=data_root # this should be where the indexed documents should be saved, but it doesn't seem to actually respect this?
    )
    indexer = Indexer(checkpoint=checkpoint_path, config=config)
    indexer.index(name=f'{doc_name}.nbits={nbits}', collection=collection_path)
    #break # uncomment this to verify it works on at least the first
end_time = time.time()
print("Took", end_time - start_time, "seconds.")

### Retrieval

In [ ]:
from colbert.data import Queries
from colbert.infra import Run, RunConfig, ColBERTConfig
from colbert import Searcher

document_retrieval_page_ranking = dict()

k = 5 # ColBERT supports up to k=100
start_time = time.time()
for doc_name, passage_list in doc_passages_dict.items():
    query_path = os.path.join(data_root, doc_name, 'questions.tsv')

    config = ColBERTConfig(
        root=data_root
    )
    searcher = Searcher(index=f'{doc_name}.nbits={nbits}', config=config)
    queries = Queries(query_path)
    document_query_retrieval_ranking[doc_name] = searcher.search_all(queries, k=k)
    #break # uncomment this to verify it works on at least the first document
end_time = time.time()
print("Took", end_time - start_time, "seconds.")

In [ ]:
# look at the rankings for the first document
first_document_rankings = pd.DataFrame(document_query_retrieval_ranking['2310.05634v2'].flat_ranking,
                                       columns = ['query_id', 'passage_id', 'rank', 'raw_score'])
first_document_rankings